# encoder-decoder-symmetric — ex1: build a tiny autoencoder whose output shape == input shape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `encoder-decoder-symmetric`. Running the final beacon cell reports progress against the `CNN: Encoder-decoder symmetric layout` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Encoder-decoder symmetric layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`encoder-decoder-symmetric`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "encoder-decoder-symmetric"
DD_SUBTOPIC = "CNN: Encoder-decoder symmetric layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Encoder-decoder symmetric layout — quick refresher

Autoencoders, U-Nets, and segmentation heads all share the same structural pattern: every encoder downsample is mirrored by a decoder upsample, so the output shape == input shape.

```
encoder:    Conv → Conv → Pool  → Conv → Conv → Pool  → ...     (spatial /= 2 per stage)
decoder:    ConvT → ConvT → Up  → ConvT → ConvT → Up  → ...     (spatial *= 2 per stage)
```

**Two upsampling Modules.** `nn.ConvTranspose2d` (learnable, can fix the checkerboard via odd kernels) and `nn.Upsample(scale_factor=2)` + a follow-up `Conv2d` (non-learnable upsample then learnable convolution — cleaner artifacts in practice). U-Net uses the Upsample+Conv variant.

**Why symmetry matters beyond just shape.** Symmetric layouts let you **skip-connect** matching encoder/decoder stages — that's how U-Net preserves spatial detail through the bottleneck. If your encoder and decoder stage counts diverge, you can't lay down those skips.

**Channel mirror.** Channel counts also mirror: encoder doubles channels per downsample (`3 → 16 → 32 → 64`), decoder halves them per upsample (`64 → 32 → 16 → 3`). Bottom-of-the-U layer keeps the highest channel count.

**Shape parity test.** A correctly-laid-out encoder-decoder Module must satisfy `model(x).shape == x.shape` for any valid input. This single assertion catches almost every off-by-one in pool/stride/padding.

### Exercise 1 — build a tiny autoencoder whose output shape == input shape

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Build a symmetric encoder-decoder Module where each encoder pool is mirrored by a decoder upsample so input shape == output shape end-to-end.
> Keywords: autoencoder, encoder, decoder, symmetric, upsample
> ```

**KCs targeted:** `encoder-downsample-stage`, `decoder-upsample-mirror-stage`

Implement `ex1_tiny_autoencoder(in_channels)` — a minimal but structurally-correct autoencoder. Spatial dims downsample by 4× (two pool stages) then upsample back by 4× (two upsample stages). Channels mirror: `C → 16 → 32` (encoder), `32 → 16 → C` (decoder).

1. Class `TinyAutoencoder(t.nn.Module)`:
   - `__init__(self, in_channels)`:
     - `super().__init__()`.
     - `self.encoder = t.nn.Sequential(`
       `   t.nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),`
       `   t.nn.ReLU(),`
       `   t.nn.MaxPool2d(2),                # spatial /= 2`
       `   t.nn.Conv2d(16, 32, kernel_size=3, padding=1),`
       `   t.nn.ReLU(),`
       `   t.nn.MaxPool2d(2),                # spatial /= 2 again`
       `)`
     - `self.decoder = t.nn.Sequential(`
       `   t.nn.Upsample(scale_factor=2),    # spatial *= 2`
       `   t.nn.Conv2d(32, 16, kernel_size=3, padding=1),`
       `   t.nn.ReLU(),`
       `   t.nn.Upsample(scale_factor=2),    # spatial *= 2 again`
       `   t.nn.Conv2d(16, in_channels, kernel_size=3, padding=1),`
       `)`
   - `forward(self, x): return self.decoder(self.encoder(x))`
2. Return an instance from `ex1_tiny_autoencoder(in_channels)`.

**Why no final ReLU in the decoder.** The output is a reconstruction in pixel space — if your inputs include negative values (e.g. zero-mean normalized images), a final ReLU clips them. A sigmoid is appropriate for `[0, 1]` images; raw linear output is appropriate for normalized images.

**Why `Upsample + Conv2d` not `ConvTranspose2d`.** ConvTranspose is learnable upsample in one shot but produces checkerboard artifacts. Upsample (nearest-neighbor) followed by a regular Conv2d is artifact-free and what U-Net actually uses.

**The shape-parity test.** The whole point of symmetric layout: `model(x).shape == x.shape` for any valid input. The test checks this on multiple input sizes.

In [ ]:
def ex1_tiny_autoencoder(in_channels: int):
    class TinyAutoencoder(t.nn.Module):
        def __init__(self, in_channels):
            super().__init__()
            self.encoder = t.nn.Sequential(
                t.nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
                t.nn.ReLU(),
                t.nn.MaxPool2d(2),
                t.nn.Conv2d(16, 32, kernel_size=3, padding=1),
                t.nn.ReLU(),
                t.nn.MaxPool2d(2),
            )
            self.decoder = t.nn.Sequential(
                t.nn.Upsample(scale_factor=2),
                t.nn.Conv2d(32, 16, kernel_size=3, padding=1),
                t.nn.ReLU(),
                t.nn.Upsample(scale_factor=2),
                t.nn.Conv2d(16, in_channels, kernel_size=3, padding=1),
            )
        def forward(self, x):
            return self.decoder(self.encoder(x))
    return TinyAutoencoder(in_channels)


<details><summary>Solution</summary>

```python
def ex1_tiny_autoencoder(in_channels: int):
    class TinyAutoencoder(t.nn.Module):
        def __init__(self, in_channels):
            super().__init__()
            self.encoder = t.nn.Sequential(
                t.nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
                t.nn.ReLU(),
                t.nn.MaxPool2d(2),
                t.nn.Conv2d(16, 32, kernel_size=3, padding=1),
                t.nn.ReLU(),
                t.nn.MaxPool2d(2),
            )
            self.decoder = t.nn.Sequential(
                t.nn.Upsample(scale_factor=2),
                t.nn.Conv2d(32, 16, kernel_size=3, padding=1),
                t.nn.ReLU(),
                t.nn.Upsample(scale_factor=2),
                t.nn.Conv2d(16, in_channels, kernel_size=3, padding=1),
            )
        def forward(self, x):
            return self.decoder(self.encoder(x))
    return TinyAutoencoder(in_channels)
```

**Why `padding=1` on every Conv2d.** With `kernel_size=3` and `padding=1`, spatial dims are preserved by the conv itself — only the explicit `MaxPool2d` / `Upsample` stages change spatial dims. This separates the two concerns: convolutions do feature mixing at fixed resolution, pool/upsample changes resolution. Without `padding=1`, every conv would shave 2 pixels off the spatial dims, making the shape arithmetic miserable.

**Why MaxPool2d for downsampling.** Three options: strided Conv, MaxPool, AvgPool. MaxPool is the canonical choice for early CNN encoders (preserves edges, cheap, no extra parameters). Strided Conv is what modern architectures use (more expressive, more parameters). AvgPool is rare in encoders but common as a final global pool before the classifier.

**The skip-connection extension.** This Module is a vanilla autoencoder — encoder output goes through a bottleneck, decoder reconstructs from the bottleneck alone. U-Net adds skip connections: each encoder stage's pre-pool activations are concatenated to the matching decoder stage's post-upsample activations. Symmetric layout is what makes those concatenations shape-compatible.

**Why a Module subclass here, not just Sequential.** Because the encoder and decoder are two named sub-pipelines we want separately introspectable (`mod.encoder`, `mod.decoder`). A flat Sequential would work for forward but lose the encoder/decoder labeling — useful for visualizing the latent, for fine-tuning just the decoder, etc.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()